# Лабораторная работа № 3. Потоковая обработка сенсорных событий в Spark Structured Streaming

**Дисциплина:** «Методы обработки больших данных» · **Этап пайплайна:** `Stream → Process`
**Среда:** Google Colab, PySpark 4.2.0, встроенный источник `rate` (Kafka не требуется)
**Максимум:** 10 баллов · **Лекция:** № 3 «Потоковые данные и распределённая обработка событий»

## Поля для заполнения

| Поле | Значение |
|---|---|
| **Фамилия, имя, отчество** | |
| **Группа** | |
| **Вариант** | |
| **Дата выполнения** | |

*(заполните таблицу и те же значения продублируйте в ячейке 1 — они попадут в отчёт)*

## Критерии оценки

| Часть | Содержание | Баллы |
|---|---|---:|
| 1 | Поток и телеметрия с `event_time`, опоздавшие события по варианту | 2 |
| 2 | Окна, агрегаты и корректный снимок результата | 2 |
| 3 | Эксперимент с watermark: два значения и измеримая метрика | 2 |
| 4 | Перенос на Kafka: что меняется в коде | 1 |
| 5 | Педагогическая часть: `TEACH CARD` и фрагмент занятия | 3 |

## Как работать

1. **Файл → Сохранить копию на Диске.** Работайте в своей копии.
2. В ячейке 1 укажите ФИО, группу и свой вариант. Параметры варианта считаются формулой —
   править их вручную нельзя.
3. Выполняйте ячейки сверху вниз. Места с пометкой `TODO` дописываете вы; до этого ячейка
   честно падает с `NotImplementedError`.
4. Поля вида **«Ваш ответ:»** и пустые ячейки таблиц заполняются текстом — они оцениваются
   так же, как код.
5. Перед сдачей запустите самопроверку в конце ноутбука.

> Все выводы должны опираться на **ваши** числа. Прогон `rate`-источника зависит от
> производительности среды, поэтому значения у всех разные — это нормально.

## Теоретический минимум

| Понятие | Коротко |
|---|---|
| `event_time` | когда событие физически произошло (ось анализа) |
| `processing_time` | когда движок его обработал |
| окно $W/S$ | интервал длиной $W$, повторяющийся через $S$; одно событие попадает примерно в $M \approx W/S$ окон |
| watermark $\Delta$ | $\mathrm{WM} = \max(t_{event}) - \Delta$; окна позади watermark финализируются, их состояние удаляется |
| состояние | «(робот, окно) → счётчики»; растёт с числом живых окон |

Компромисс, который вы измерите в части 3: **больше $\Delta$ — полнее данные, но больше
состояния и позже готов итог.**

Подробности — в лекции № 3 и в `README_lab_03.md`.

---
# Часть 1. Поток и телеметрия (2 балла)

## Ячейка 1. Вариант и параметры (дана готовой)

Параметры вычисляются формулой: группа `(n−1)//5` задаёт нагрузку, число роботов и частоту
опоздавших событий, позиция `(n−1)%5` — геометрию окна, сдвиг опоздавших событий,
дополнительную метрику и аудиторию для методической части. Заполните три строки вверху.

In [ ]:
# !pip -q install "pyspark==4.2.0"      # в Colab раскомментируйте эту строку
import json
import time
from typing import Dict, List

# =========== ЗАПОЛНИТЕ ===========
STUDENT_NAME  = "Иванов Иван Иванович"      # <-- ваши ФИО
STUDENT_GROUP = "группа"                   # <-- ваша группа
VARIANT       = 1                          # <-- ваш вариант
# =================================
assert 1 <= VARIANT <= 25, "Вариант — целое число от 1 до 25"


def variant_params(n: int) -> dict:
    """Параметры варианта по таблице задания.

    Группа вариантов (n−1)//5 задаёт нагрузку и число роботов,
    позиция внутри группы (n−1)%5 — геометрию окна, долю и сдвиг опоздавших событий.
    """
    rows_per_sec = [20, 30, 40, 50, 60][(n - 1) // 5]
    robots       = [3, 4, 5, 6, 8][(n - 1) // 5]
    late_every   = [11, 17, 19, 23, 29][(n - 1) // 5]     # каждое k-е событие опаздывает
    window_s, slide_s = [(10, 5), (20, 5), (20, 10), (30, 10), (15, 5)][(n - 1) % 5]
    late_shift_s = [15, 20, 25, 30, 35][(n - 1) % 5]      # на сколько сдвинут event_time
    extra_metric = ["vibration_rms", "temp_spread", "temp_stddev",
                    "hot_share", "vibration_p95"][(n - 1) % 5]
    audience     = ["5–7 класс", "8–9 класс", "10–11 класс",
                    "кружок робототехники", "СПО"][(n - 1) % 5]
    return {
        "rows_per_sec": rows_per_sec,
        "robots": robots,
        "window_s": window_s,
        "slide_s": slide_s,
        "late_every": late_every,
        "late_shift_s": late_shift_s,
        "watermark_short": "10 seconds",
        "watermark_long": f"{late_shift_s + 10} seconds",   # заведомо больше сдвига
        "extra_metric": extra_metric,
        "audience": audience,
    }


CFG = variant_params(VARIANT)
RUN_SECONDS = 15        # длительность одного прогона потока

print(f"Студент : {STUDENT_NAME}, группа {STUDENT_GROUP}")
print(f"Вариант : {VARIANT}")
for key, value in CFG.items():
    print(f"  {key:<16}: {value}")
print(f"\nОкон на одно событие (теория): M ≈ W/S = "
      f"{CFG['window_s']}/{CFG['slide_s']} = {CFG['window_s'] / CFG['slide_s']:.0f}")
print(f"Ожидаемая нагрузка: {CFG['rows_per_sec']} событий/с, "
      f"доля опоздавших ≈ {100 / CFG['late_every']:.1f} %")



def todo(step):
    raise NotImplementedError(f"Не выполнен шаг TODO {step} — допишите код в отмеченном месте")

## Ячейка 2. Сессия Spark

**TODO 2.** Создайте `SparkSession` в локальном режиме.

Подсказки: мастер `local[*]` (потоку нужно минимум два потока выполнения — один занимает
приёмник источника); `spark.sql.shuffle.partitions = 4`, иначе Spark создаст 200 почти пустых
партиций и каждый micro-batch будет длиться лишние секунды; `setLogLevel("ERROR")` уберёт шум
в выводе. В конце напечатайте версию Spark и число ядер.

In [ ]:
# !pip -q install "pyspark==4.2.0"     # в Colab раскомментируйте
from pyspark.sql import SparkSession, functions as F

# TODO 2: создайте сессию spark и напечатайте её параметры
todo("2")

## Ячейка 3. Телеметрия робота

**TODO 3.** Напишите функцию `build_telemetry(source, cfg)`, которая превращает поток `rate`
(колонки `timestamp`, `value`) в телеметрию:

| Колонка | Как считается |
|---|---|
| `robot_id` | `"robot-" + (value % cfg["robots"])` |
| `is_late` | `value % cfg["late_every"] == 0` — «правда» эксперимента: какое событие опоздало |
| `event_time` | для опоздавших `timestamp − cfg["late_shift_s"] секунд`, иначе `timestamp` |
| `motor_temp` | `43.0 + (value % 25) * 0.55` |
| `vibration` | `abs(sin(value / 8))` |

Сначала проверьте логику на **конечной** таблице `spark.range(0, 12)` с колонкой
`timestamp = current_timestamp()`: в потоке искать ошибку в выражении гораздо дороже.
Напечатайте таблицу и число опоздавших событий в примере.

**Важно:** сдвигать нужно `event_time` в **прошлое**. Событие «из будущего» watermark не
отбросит, и эксперимент части 3 не получится.

In [ ]:
def build_telemetry(source, cfg: dict):
    """Превратить сырой поток rate в телеметрию роботов."""
    # TODO 3
    todo("3")


# TODO 3: проверка на конечной таблице spark.range(0, 12)
todo("3")

---
# Часть 2. Окна, агрегаты и снимок (2 балла)

## Ячейка 4. Запуск потока с окнами

**TODO 4.** Напишите функцию `run_stream(cfg, watermark, seconds, name)`:

1. источник `rate` с `rowsPerSecond = cfg["rows_per_sec"]` и `numPartitions = 2`;
2. `build_telemetry`;
3. `withWatermark("event_time", watermark)`;
4. группировка по `window(event_time, "{W} seconds", "{S} seconds")` и `robot_id`;
5. агрегаты: `events` (счёт), `late_events` (сумма `is_late`), `avg_temp`, `max_temp` и
   дополнительная метрика вашего варианта (`cfg["extra_metric"]`);
6. запись в `memory`-приёмник, `outputMode("update")`, триггер 1 секунда;
7. ожидание `seconds` секунд, сохранение `list(query.recentProgress)`, затем `query.stop()`.

**Не используйте `query.processAllAvailable()`.** Источник `rate` бесконечен: «все доступные
данные» у него не заканчиваются, и вызов может заблокировать ячейку надолго. Останавливайте
поток по таймеру.

Функция возвращает `(таблица обновлений, список метрик батчей)`.

Выражения дополнительных метрик:

```python
AGGREGATES = {
    "vibration_rms": F.sqrt(F.avg(F.pow("vibration", 2))).alias("extra_metric"),
    "temp_spread":   (F.max("motor_temp") - F.min("motor_temp")).alias("extra_metric"),
    "temp_stddev":   F.stddev_samp("motor_temp").alias("extra_metric"),
    "hot_share":     F.avg((F.col("motor_temp") > 50).cast("double")).alias("extra_metric"),
    "vibration_p95": F.percentile_approx("vibration", 0.95).alias("extra_metric"),
}
```

In [ ]:
AGGREGATES = {
    "vibration_rms": F.sqrt(F.avg(F.pow("vibration", 2))).alias("extra_metric"),
    "temp_spread":   (F.max("motor_temp") - F.min("motor_temp")).alias("extra_metric"),
    "temp_stddev":   F.stddev_samp("motor_temp").alias("extra_metric"),
    "hot_share":     F.avg((F.col("motor_temp") > 50).cast("double")).alias("extra_metric"),
    "vibration_p95": F.percentile_approx("vibration", 0.95).alias("extra_metric"),
}


def run_stream(cfg: dict, watermark: str, seconds: int = RUN_SECONDS, name: str = "robot_windows"):
    """Запустить поток на заданное время и вернуть (обновления, метрики прогресса)."""
    # TODO 4
    todo("4")


raw_updates, progress_short = run_stream(CFG, CFG["watermark_short"], name="w_short")
print("Строк в memory-sink:", raw_updates.count())
print("Уникальных пар (окно, робот):", raw_updates.select("window", "robot_id").distinct().count())

**Ваш ответ (0,5 балла части 2).** Почему эти два числа различаются? Что именно хранит
`memory`-приёмник в режиме `update`?

> *(впишите 2–3 предложения)*

## Ячейка 5. Снимок состояния окон

**TODO 5.** Постройте функцию `snapshot(updates)`, которая оставляет по одной строке на пару
«окно — робот» — ту, где число событий максимально (это и есть последнее состояние окна).

Подсказка: оконная функция
`row_number().over(Window.partitionBy("window", "robot_id").orderBy(F.col("events").desc()))`,
затем фильтр `rn = 1`. Выведите `window_start`, `window_end`, `robot_id`, `events`,
`late_events`, округлённые `avg_temp`, `max_temp` и вашу дополнительную метрику, отсортировав
по убыванию начала окна.

In [ ]:
from pyspark.sql import Window


def snapshot(updates):
    """Последнее состояние каждого окна."""
    # TODO 5
    todo("5")


result_short = snapshot(raw_updates)
result_short.show(12, truncate=False)
print("Окон в снимке:", result_short.count())
print("Событий учтено в окнах:", result_short.agg(F.sum("events")).first()[0])

---
# Часть 3. Метрики и эксперимент с watermark (2 балла)

## Ячейка 6. Метрики micro-batch

**TODO 6.** Напишите `progress_summary(progress)`, которая по списку `recentProgress`
возвращает словарь: число батчей с данными, суммарное число входных событий, пропускную
способность (события / суммарное время `durationMs["triggerExecution"]`), среднюю длительность
батча, максимальное число строк состояния и память состояния
(`stateOperators[0]["numRowsTotal"]`, `["memoryUsedBytes"]`) и, главное, суммарное
`numRowsDroppedByWatermark` — число событий, отброшенных по watermark.

Выведите сводку и блок `progress[-1]["eventTime"]`: в нём видно, на сколько watermark отстаёт
от максимального времени события.

In [ ]:
def progress_summary(progress):
    """Свести метрики micro-batch к показателям из лекции."""
    # TODO 6
    todo("6")


summary_short = progress_summary(progress_short)
for key, value in summary_short.items():
    print(f"  {key:<32}: {value}")

## Ячейка 7. Два значения watermark

**TODO 7.** Повторите прогон с `CFG["watermark_long"]` (он заведомо больше сдвига опоздавших
событий вашего варианта) и сведите оба результата в таблицу: входные события, пропускная
способность, длительность батча, строки и память состояния, отброшенные события, число окон в
снимке и сумма `late_events`.

In [ ]:
import pandas as pd

# TODO 7: второй прогон и сравнительная таблица
todo("7")

**Ваш ответ (1 балл части 3).** Заполните по своим числам:

| Показатель | watermark короткий | watermark длинный |
|---|---:|---:|
| событий отброшено watermark | | |
| строк состояния (максимум) | | |
| опоздавших событий учтено в окнах | | |

1. Что изменилось при увеличении watermark и **чем за это заплатили**?
   > *(ваш ответ)*
2. Какой эффект этот прогон измерить **не позволяет** и почему?
   > *(ваш ответ)*

## Ячейка 8. Кратность окон

**TODO 8.** Сравните фактическое число попаданий событий в окна (сумма `events` в снимке) с
числом входных событий. Отношение должно быть близко к $M \approx W/S$ вашего варианта.
Напечатайте оба числа и объясните расхождение.

In [ ]:
# TODO 8: фактическая и теоретическая кратность окон
todo("8")

**Ваш ответ.** Фактическая кратность ______ при теоретической ______. Расхождение
объясняется:

> *(ваш ответ)*

---
# Часть 4. Перенос на Kafka (1 балл)

Код ниже **не выполняется**: broker в работе не нужен. Разберите его и ответьте на вопросы.

In [ ]:
kafka_code = """
raw = (
    spark.readStream
    .format("kafka")
    .option("kafka.bootstrap.servers", "broker:9092")
    .option("subscribe", "robot.telemetry")
    .option("startingOffsets", "latest")
    .option("maxOffsetsPerTrigger", 5000)
    .load()
)

schema = "robot_id STRING, event_time TIMESTAMP, motor_temp DOUBLE, vibration DOUBLE"
telemetry = (
    raw.select(F.from_json(F.col("value").cast("string"), schema).alias("e"))
       .select("e.*")
)
"""
print(kafka_code)

**Ваши ответы (1 балл).**

| Вопрос | Ваш ответ |
|---|---|
| Какие строки вашего кода нужно заменить при переходе на Kafka и какие остаются без изменений? | |
| Что такое offset и почему это **не** временная метка? | |
| Как `key = robot_id` влияет на порядок событий и на параллелизм? | |
| Зачем в потоковом запросе с Kafka обязателен `checkpointLocation`? | |

---
# Часть 5. Педагогическая часть (3 балла)

Инженерный результат работы — три утверждения, которые можно объяснить школьнику без слов
«watermark» и «micro-batch»:

1. **У события есть два времени** — когда оно случилось и когда о нём узнали; считать надо по
   первому.
2. **Ждать опоздавших — значит занимать память**: чем дольше система ждёт, тем больше хранит и
   тем позже выдаёт итог.
3. **Скользящее окно считает одно событие несколько раз** — ровно $W/S$ раз, и это свойство
   метода, а не ошибка.

## 5.1. `TEACH CARD` (1,5 балла)

Аудитория задана вашим вариантом (`CFG["audience"]`), числа берутся из **вашего** прогона.

| Поле | Содержание |
|---|---|
| **Название учебного проекта** | |
| **Целевая аудитория** | *(из варианта)* |
| **Исследовательский вопрос** | |
| **Источник данных** | |
| **Инженерная концепция, сохраняемая без упрощения** | |
| **Что упрощается относительно университетской лабораторной** | |
| **Алгоритм действий обучающегося (4–6 шагов)** | |
| **Измеримый результат / метрика** | |
| **Критерий успешного выполнения** | |
| **Вариант усложнения для НТО / хакатона** | |
| **Риски и ограничения** | |

## 5.2. Фрагмент занятия на 15–20 минут (1,5 балла)

Выберите **один** из трёх выводов и опишите фрагмент. Обязательны: действия учеников (а не
рассказ учителя), **два числа вашего варианта**, ключевой вопрос и способ проверки понимания.

| Пункт | Ваш ответ |
|---|---|
| Вывод (1, 2 или 3) | |
| Класс и формат | |
| Время | |
| Материалы | |
| Числа вашего прогона и как они используются | |

**Ход фрагмента (3–5 шагов, кто что делает):**

1. …
2. …
3. …

**Ключевой вопрос для учеников:** …

**Проверка понимания (как узнаете, что вывод усвоен; какое заблуждение ловится):** …

**Критерии.** 1,5 балла — `TEACH CARD` заполнен полностью, аудитория соответствует варианту,
инженерная концепция не искажена. 1,5 балла — фрагмент реализуем за 15–20 минут, содержит
действия учеников, числа своего прогона, ключевой вопрос и проверку понимания.

Не засчитывается: пересказ лекции без действий учеников; чужие числа; объяснение термина
«watermark» словом «watermark».

---
## Самопроверка перед сдачей

In [ ]:
ANSWERS = {
    "dropped_short": None,      # событий отброшено watermark при коротком watermark
    "dropped_long": None,       # то же при длинном watermark
    "state_rows_short": None,   # строк состояния (максимум) при коротком
    "state_rows_long": None,    # то же при длинном
    "windows_short": None,      # окон в снимке при коротком watermark
    "multiplicity": None,       # фактическая кратность окон
}

checks = {
    "Заполнены ФИО и группа":        STUDENT_NAME != "Иванов Иван Иванович",
    "Вариант 1..25":                 1 <= VARIANT <= 25,
    "Телеметрия построена":          "raw_updates" in dir() and raw_updates.count() > 0,
    "Снимок окон получен":           "result_short" in dir() and result_short.count() > 0,
    "Снимок короче истории":         "result_short" in dir() and result_short.count() <= raw_updates.count(),
    "Метрики посчитаны":             "summary_short" in dir(),
    "Эксперимент с watermark":       all(ANSWERS[k] is not None for k in ("dropped_short", "dropped_long")),
    "Состояние измерено":            all(ANSWERS[k] is not None for k in ("state_rows_short", "state_rows_long")),
    "Кратность окон оценена":        ANSWERS["multiplicity"] is not None,
    "Длинный watermark даёт меньше потерь":
        (ANSWERS["dropped_long"] or 0) <= (ANSWERS["dropped_short"] or 0),
}
for name, ok in checks.items():
    print(f"{'OK ' if ok else 'НЕТ'}  {name}")
print(f"\nВыполнено: {sum(checks.values())} из {len(checks)}")
print("\nПедагогическая часть (3 балла) проверяется преподавателем вручную.")

## Перед отправкой

- [ ] заполнена таблица с ФИО, группой, вариантом и датой;
- [ ] `VARIANT` совпадает с номером в списке, параметры не правились руками;
- [ ] выполнены все ячейки, выводы сохранены;
- [ ] заполнены поля «Ваш ответ» в частях 2, 3 и 4;
- [ ] заполнен словарь `ANSWERS`;
- [ ] `TEACH CARD` заполнен полностью, аудитория соответствует варианту;
- [ ] фрагмент занятия содержит действия учеников, два числа вашего прогона, ключевой вопрос и
      проверку понимания;
- [ ] самопроверка показывает «Выполнено: 10 из 10».